# Notebook 3 (Fase 2) — API REST con FastAPI + MongoDB + ngrok
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:** Levantar una API que LEE de MongoDB (no recalcula) y expone 3 endpoints REST via ngrok.

**Prerequisitos:** `NGROK_AUTHTOKEN` en Secrets de Colab. Notebooks 1 y 2 ejecutados (colecciones `precios_ohlcv`, `predicciones` y `metricas_modelos` pobladas).

**Endpoints:**
- `GET /api/salud` — estado del servidor y de la base de datos
- `GET /api/mercado/{ticker}` — datos OHLCV con indicadores tecnicos
- `GET /api/svc/{ticker}` — senal y metricas del clasificador SVC

**Pipeline:** MongoDB → FastAPI (lee, no calcula) → ngrok → **API publica en Internet**

In [ ]:
# Paso 1 — Instalar librerias necesarias
!pip install fastapi uvicorn pyngrok nest-asyncio "pymongo[srv]" --quiet

print("Librerias instaladas correctamente")

In [ ]:
# Paso 2 — Configurar credenciales: ngrok (Secrets) y MongoDB Atlas
# El authtoken de ngrok se lee de los Secrets de Colab, tal como pide la
# especificacion (nunca se escribe en texto plano en el notebook).
# La password de MongoDB se pide de forma oculta con getpass, siguiendo el
# mismo patron de conexion usado en los Notebooks 1 y 2.
from google.colab import userdata
from getpass import getpass
from pyngrok import conf
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# --- ngrok: pyngrok 8.x usa conf.get_default().auth_token ---
conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")

# --- MongoDB Atlas ---
DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi("1"))

try:
    client.admin.command("ping")
    print("ngrok y MongoDB configurados correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]

In [ ]:
# Paso 3 — Crear la app FastAPI con CORS habilitado + endpoint /api/salud
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Ernesto Investing AI - API")

# CORS habilitado para que el frontend (GitHub Pages) pueda hacer fetch() a esta API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/salud")
def salud():
    """Verifica que el servidor y la base de datos esten operativos."""
    try:
        client.admin.command("ping")
        n_precios = db["precios_ohlcv"].count_documents({})
        n_pred = db["predicciones"].count_documents({})
        return {
            "estado": "operativo",
            "base_datos": "conectada",
            "documentos_precios": n_precios,
            "documentos_predicciones": n_pred
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("App FastAPI creada. Endpoint /api/salud listo")

In [ ]:
# Paso 4 — Endpoint de datos de mercado (lee precios_ohlcv de MongoDB, no recalcula)
@app.get("/api/mercado/{ticker}")
def mercado(ticker: str):
    ticker = ticker.upper()

    # Se excluyen _id (no serializable a JSON de forma directa) y created_at (uso interno)
    docs = list(db["precios_ohlcv"].find(
        {"ticker": ticker}, {"_id": 0, "created_at": 0}
    ).sort("fecha", 1))

    if not docs:
        raise HTTPException(status_code=404, detail=f"No hay datos para {ticker}")

    return {"ticker": ticker, "cantidad": len(docs), "datos": docs}

print("Endpoint /api/mercado/{ticker} creado")

In [ ]:
# Paso 5 — Endpoint de prediccion SVC (lee predicciones y metricas_modelos de MongoDB)
@app.get("/api/svc/{ticker}")
def svc(ticker: str):
    ticker = ticker.upper()

    prediccion = db["predicciones"].find_one(
        {"ticker": ticker, "modelo": "SVC"}, {"_id": 0, "created_at": 0}
    )
    metricas = db["metricas_modelos"].find_one(
        {"ticker": ticker, "modelo": "SVC"}, {"_id": 0, "fecha_entrenamiento": 0}
    )

    if not prediccion:
        raise HTTPException(status_code=404, detail=f"No hay prediccion SVC para {ticker}")

    return {"ticker": ticker, "prediccion": prediccion, "metricas": metricas}

print("Endpoint /api/svc/{ticker} creado")

In [ ]:
# Paso 6 — Levantar el servidor uvicorn + exponerlo a Internet con ngrok
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok

nest_asyncio.apply()

# Se cierra cualquier tunel previo para evitar conflictos al re-ejecutar la celda
ngrok.kill()
tunel = ngrok.connect(8000)

print("=" * 65)
print(f"URL PUBLICA DE LA API: {tunel.public_url}")
print("=" * 65)
print(f"Salud:    {tunel.public_url}/api/salud")
print(f"Mercado:  {tunel.public_url}/api/mercado/BVN")
print(f"SVC:      {tunel.public_url}/api/svc/BVN")
print(f"Swagger:  {tunel.public_url}/docs")
print("=" * 65)
print()
print(">>> COPIA LA URL PUBLICA <<<")
print(">>> La pegaras en el archivo index.html del frontend <<<")

def correr():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=correr, daemon=True).start()
print("\nAPI operativa. Manten este notebook ejecutandose.")

In [ ]:
# Paso 7 — Celda de verificacion final: probar los 3 endpoints desde el propio notebook
import time
import requests

time.sleep(2)  # margen para que uvicorn termine de levantar en el hilo en segundo plano
headers = {"ngrok-skip-browser-warning": "true"}

endpoints = {
    "Salud": f"{tunel.public_url}/api/salud",
    "Mercado (BVN)": f"{tunel.public_url}/api/mercado/BVN",
    "SVC (BVN)": f"{tunel.public_url}/api/svc/BVN",
}

print("Verificando endpoints de la API...")
print()

todos_ok = True
for nombre, url in endpoints.items():
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            print(f"  [OK] {nombre}: {r.status_code}")
        else:
            print(f"  [ATENCION] {nombre}: {r.status_code} — {r.text[:100]}")
            todos_ok = False
    except Exception as e:
        print(f"  [ERROR] {nombre}: {e}")
        todos_ok = False

print()
if todos_ok:
    print("Verificacion exitosa: los 3 endpoints responden correctamente.")
else:
    print("Atencion: revisar los endpoints marcados arriba antes de conectar el frontend.")

## Paso 8 — Conectar el frontend

1. **Copia la URL pública** que se mostró en el Paso 6.
2. Abre `index.html` del frontend.
3. Pega la URL en el campo de configuración.
4. Sube el frontend a GitHub Pages.
5. ¡El frontend muestra datos reales!

**Nota:** En ngrok gratuito, la primera visita muestra una página de advertencia. Los archivos HTML del frontend incluyen el header `ngrok-skip-browser-warning` para evitarla en las llamadas `fetch()`.

**Pipeline completo:** yfinance → MongoDB → SVC → MongoDB → FastAPI → Frontend HTML ✓